# Impact of Spatial-Derived Variables on Jeonse Price Prediction in Buk-gu, Pohang

**Course:** 기계학습개론 5교시 분반  
**Team:** 호재호  
**Members:** 22100555 이재호 · 22100372 서호재 · 22200392 손현건 · 22300533 이세리 · 22400264 박서연

This notebook builds a machine-learning workflow to test whether spatial-derived variables improve Jeonse price prediction in Buk-gu, Pohang. It combines the dataset construction plan, literature-based research gap, methodology, model implementation, and result interpretation in one Colab-ready file.

## 1. Research Objective

This study examines whether neighborhood-level spatial variables improve the prediction of Jeonse prices in Buk-gu, Pohang.

The core research question is:

> Do spatial-derived variables such as school infrastructure, park accessibility, and coastal proximity provide additional predictive power beyond basic apartment transaction and structural variables?

The expected contribution is to move beyond basic apartment characteristics by adding local context variables that represent environmental and educational conditions.

## 2. Dataset Construction Status

### 2.1 Housing Transaction Data

| Item | Description |
|---|---|
| Source | Ministry of Land, Infrastructure and Transport (MOLIT), Real Estate Transaction Management System |
| Address | https://rt.molit.go.kr/pt/xls/xls.do?&mobileAt= |
| Collected data | Apartment lease transaction records for Buk-gu, Pohang |
| Main variables | sigungu, apartment complex name, lease type, exclusive area, contract year-month, contract day, deposit, monthly rent, floor, year built, road name, contract type, renewal request status |
| Purpose | Core transaction dataset for predicting Jeonse prices |

Jeonse transactions are the primary target for modeling. If additional data are required, monthly rent transactions can be converted into Jeonse-equivalent deposits and used as an expanded-sample specification.

### 2.2 Deposit-to-Rent Conversion Rate

The conversion rate can be calculated as:

$$
\text{Conversion Rate (%)} =
\frac{\text{Annual Rent}}{\text{Original Deposit} - \text{New Deposit}} \times 100
$$

Example:

- Original Jeonse deposit: 300 million KRW
- New monthly-rent deposit: 50 million KRW
- Monthly rent: 1 million KRW
- Annual rent: 12 million KRW
- Deposit difference: 250 million KRW
- Conversion rate: \(12 / 250 \times 100 = 4.8\%\)

For prediction with monthly-rent transactions, the code uses the decimal form of the conversion rate. If the conversion rate is 6.5%, then \(r = 0.065\):

$$
\text{Jeonse-equivalent Deposit} =
\text{Monthly-rent Deposit} +
\frac{\text{Monthly Rent} \times 12}{r}
$$

In this notebook, `CONVERSION_RATE = 0.065` means 6.5% expressed as a decimal. Results from the converted monthly-rent sample should be interpreted as predictions of **Jeonse-equivalent deposit**, not observed Jeonse-only deposit.

### 2.3 School Data

| Item | Description |
|---|---|
| Source | Gyeongsangbuk-do Office of Education |
| Address | https://www.gbe.kr/ |
| Collected data | School status information for elementary, middle, and high schools in Pohang |
| Raw variables | school name, school level, address, establishment type, contact information |
| Derived variables | number of elementary schools by dong, number of middle schools by dong, number of high schools by dong, total number of schools by dong, school-count dummy variables, dummy indicating whether all three school levels are present |
| Purpose | Capture local educational infrastructure and test whether education-related neighborhood conditions improve prediction |

Implementation boundary: the current modeling sheet contains school count variables and `동_초중고모두있음여부`. Because the provided text does not define thresholds for school-count range dummy variables, this notebook does not create additional range dummies.

### 2.4 Park Data

| Item | Description |
|---|---|
| Source | Park-related public data / Public Data Portal |
| Address | https://www.data.go.kr/ |
| Collected data | Park existence, park counts, and park area information for Buk-gu, Pohang |
| Derived variables | park presence, number of parks, maximum park area, total park area |
| Threshold | near 500m |
| Purpose | Reflect green-space accessibility and residential environmental quality |

### 2.5 Sea-Related Spatial Data

| Item | Description |
|---|---|
| Source | Administrative-area-based spatial classification for Buk-gu, Pohang |
| Reference | Calculate whether the sea is within a 500m straight-line distance from the apartment |
| Derived variable | sea proximity dummy |
| Definition | 1 for coastal administrative areas or areas within 500m of the coastline; 0 otherwise |
| Purpose | Capture coastal accessibility, which may affect local housing preference and Jeonse price formation |

Implementation boundary: this notebook uses the already-derived park and sea variables in the modeling sheet. It does not recompute GIS distances or verify the 500m threshold from raw coordinates, because raw coordinate data and distance-calculation procedures are not included in the provided notebook input.

## 3. Derived Dataset Structure

The final dataset is organized in three stages.

1. **Raw Housing Dataset**  
   Apartment lease transaction data for Buk-gu, Pohang were collected and cleaned.

2. **Integrated Dataset**  
   Housing transactions were merged with school, park, and sea-related spatial variables.

3. **Modeling Dataset**  
   A separate modeling sheet was constructed using only the variables required for prediction analysis:
   - transaction variables
   - housing attributes
   - location variables
   - spatially derived variables

Expected outcomes:

- Compare a baseline model and a spatially extended model.
- Identify which spatial features matter most, such as parks, schools, or sea proximity.
- Better explain the local housing market structure in Buk-gu, Pohang.
- Improve the realism of housing price prediction beyond basic apartment characteristics.

## 4. Literature Review and Research Gap

| Theme | Study | Method | Key point | Limitation |
|---|---|---|---|---|
| Feature categorization & interpretability | Na & Park (2025) | XAI-based model | Variables categorized into structural, locational, and educational groups; education features identified as important | No explicit modeling of spatial relationships |
| Machine-learning approach | Na & Kim (2019) | MLR, Random Forest, SVM with RMSE comparison | Feature engineering improves prediction; dummy variables enhance performance | Lack of spatially derived features |
| Spatial analysis approach | Hong & Lee (2015) | GWR + time-series analysis | Housing prices show spatial dependence; determinants vary across regions and time | Spatial dependence is acknowledged but not translated into ML feature engineering |

### Key Insight

Prior research suggests that spatial factors matter in housing price analysis, but area-level spatial features such as school counts, park accessibility, and natural environment indicators are still insufficiently incorporated into machine-learning prediction models.

### Final Research Gap

There is a lack of studies that:

- incorporate area-level spatially derived variables,
- reflect local spatial context,
- apply the model beyond metropolitan areas such as Seoul.

Therefore, this project focuses on **area-level spatial features capturing local context in housing price prediction** for Buk-gu, Pohang.

## 5. Methodology

The methodology consists of five steps:

1. **Data preprocessing**
   - Check missing values and outliers.
   - Use Jeonse transactions as the primary modeling target.
   - Convert monthly rent transactions into Jeonse-equivalent deposits only when using the expanded-sample specification.
   - Standardize numerical variables with `StandardScaler`.
   - Convert categorical or binary variables into encoded variables when needed.
   - Check multicollinearity among numerical variables using VIF.

2. **Spatial-derived variable construction**
   - Sea proximity dummy.
   - Park presence, park count, maximum park area, total park area.
   - Elementary, middle, high school counts and school-level availability dummy.

3. **Feature set design**
   - Group A: baseline apartment and transaction variables.
   - Group B: Group A plus spatial-derived variables.

4. **Model implementation**
   - Ridge Regression.
   - Random Forest Regressor.
   - XGBoost Regressor.

5. **Model evaluation and interpretation**
   - RMSE, MAE, and R².
   - Feature importance for tree-based models.
   - Optional SHAP analysis for detailed interpretation.

Implementation note: this notebook also includes clearly labeled supplementary checks, such as an `읍면동` location-control comparison, log-target modeling, contract-type control, and Jeonse-only robustness. These checks are not part of the original two-group methodology; they are included to test whether the main conclusion is sensitive to modeling choices.

## 6. Colab Setup

Run the next cell first. It clones the GitHub repository if the dataset is not already available in the runtime.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import importlib.util

REPO_URL = "https://github.com/hyeon03-sketch/IML-Final-project.git"
REPO_DIR = Path("IML-Final-project")
DATA_FILE = Path("IML_Final_dataset.xlsx")

if not DATA_FILE.exists() and Path("../IML_Final_dataset.xlsx").exists():
    os.chdir("..")

if not DATA_FILE.exists():
    if not REPO_DIR.exists():
        subprocess.check_call(["git", "clone", REPO_URL])
    os.chdir(REPO_DIR)

print("Working directory:", Path.cwd())
print("Dataset exists:", Path("IML_Final_dataset.xlsx").exists())

packages = {
    "pandas": "pandas",
    "numpy": "numpy",
    "openpyxl": "openpyxl",
    "sklearn": "scikit-learn",
    "xgboost": "xgboost",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
}
missing = [pip_name for import_name, pip_name in packages.items() if importlib.util.find_spec(import_name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("Installed:", missing)
else:
    print("All required packages are already installed.")

## 7. Imports and Constants

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
RND = 42
DATA_PATH = Path("IML_Final_dataset.xlsx")
SHEET_NAME = "최종데이터셋_모델용"
TARGET = "보증금_환산(만원)"
CONVERSION_RATE = 0.065
OUT_DIR = Path("outputs_colab")
OUT_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

## 8. Load Data and Build Target

The notebook constructs a Jeonse-equivalent target in 10,000 KRW units. Jeonse transactions use the original deposit. Monthly rent transactions are converted with the 6.5% conversion rate only for the expanded-sample analysis. A Jeonse-only robustness check is included later so that the conversion assumption does not silently drive the conclusion.

In [ ]:
df = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)

is_wolse = df["전월세구분"].eq("월세")
df[TARGET] = df["보증금(만원)"].astype(float)
df.loc[is_wolse, TARGET] = (
    df.loc[is_wolse, "보증금(만원)"]
    + df.loc[is_wolse, "월세금(만원)"] * 12.0 / CONVERSION_RATE
)
df = df[df[TARGET] > 0].copy()

print("Shape:", df.shape)
print("Missing cells:", int(df.isna().sum().sum()))
print("Dong count:", df["읍면동"].nunique())
print("Contract years:", sorted(df["계약연도"].unique()))

display(df["전월세구분"].value_counts().to_frame("count"))
display(df[[TARGET, "전용면적(㎡)", "층", "건물연령(계약기준)", "계약연도", "계약월"]].describe())
display(df.head())

## 9. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df[TARGET], bins=60, ax=axes[0], color="#2563eb")
axes[0].set_title("Jeonse-equivalent deposit")
axes[0].set_xlabel("10,000 KRW")

sns.histplot(np.log1p(df[TARGET]), bins=60, ax=axes[1], color="#059669")
axes[1].set_title("log1p target")
axes[1].set_xlabel("log1p(10,000 KRW)")

plt.tight_layout()
plt.savefig(OUT_DIR / "target_distribution.png", dpi=150)
plt.show()

corr_cols = [
    TARGET, "전용면적(㎡)", "층", "건물연령(계약기준)", "계약연도", "계약월",
    "공원수", "최대공원면적(㎡)", "총공원면적(㎡)", "바다여부",
    "동_초등학교수", "동_중학교수", "동_고등학교수", "동_총학교수",
]
plt.figure(figsize=(11, 8))
sns.heatmap(df[corr_cols].corr(), cmap="RdBu_r", center=0, annot=True, fmt=".2f")
plt.title("Correlation Matrix")
plt.tight_layout()
plt.savefig(OUT_DIR / "correlation_matrix.png", dpi=150)
plt.show()

## 10. Multicollinearity Check with VIF

VIF is checked for numerical features. This matters because some derived variables can overlap mechanically. In particular, `동_총학교수` is the sum of elementary, middle, and high school counts, so it is not used together with those three count variables in the default model feature set. It remains in this VIF check only to document the redundancy.

In [ ]:
def calculate_vif(data, columns):
    X = data[columns].astype(float).replace([np.inf, -np.inf], np.nan).dropna()
    rows = []
    for col in columns:
        y = X[col].to_numpy()
        others = [c for c in columns if c != col]
        X_other = X[others].to_numpy()
        X_design = np.column_stack([np.ones(len(X_other)), X_other])
        beta, *_ = np.linalg.lstsq(X_design, y, rcond=None)
        pred = X_design @ beta
        ss_res = np.sum((y - pred) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
        vif = np.inf if r2 >= 0.999999 else 1 / (1 - r2)
        rows.append({"feature": col, "vif": vif})
    return pd.DataFrame(rows).sort_values("vif", ascending=False)

vif_cols = [
    "전용면적(㎡)", "층", "건물연령(계약기준)", "계약연도", "계약월",
    "공원수", "최대공원면적(㎡)", "총공원면적(㎡)",
    "동_초등학교수", "동_중학교수", "동_고등학교수", "동_총학교수",
]
display(calculate_vif(df, vif_cols))

## 11. Feature Set Design

The main comparison follows the original methodology:

- **Group A (`A_baseline`)**: baseline apartment and transaction variables.
- **Group B (`B_spatial_extended`)**: Group A plus spatial-derived variables.

To avoid a clear multicollinearity problem, the default spatial feature set uses elementary, middle, and high school counts separately, but excludes `동_총학교수` because it is the sum of those three variables.

The construction plan mentions school-count range dummy variables. They are not generated here because the provided content does not specify the range thresholds, and the current modeling sheet does not include such range-dummy columns. The notebook therefore uses only the available school count variables and `동_초중고모두있음여부`.

Supplementary location-control checks are also included but are not part of the original Group A/B design:

- **`A_location_control_with_dong`**: Group A plus administrative dong (`읍면동`).
- **`B_spatial_location_control_with_dong`**: Group B plus administrative dong (`읍면동`).

Reason for the supplementary check: if `읍면동` is included, it may absorb local spatial differences that overlap with school, park, or sea variables. Therefore, the notebook reports both the original feature-set comparison and the additional location-control comparison separately.

In [ ]:
HOUSE_TIME = ["전용면적(㎡)", "층", "건물연령(계약기준)", "계약연도", "계약월"]
DONG = ["읍면동"]
SEA = ["바다여부"]
SPATIAL = [
    "공원여부", "공원수", "최대공원면적(㎡)", "총공원면적(㎡)",
    "동_초등학교수", "동_중학교수", "동_고등학교수",
    "동_초중고모두있음여부",
]

BINARY_COLS = {"바다여부", "공원여부", "동_초중고모두있음여부"}
CATEGORICAL_COLS = {"읍면동", "전월세구분"}

MAIN_FEATURE_GROUPS = {
    "A_baseline": HOUSE_TIME,
    "B_spatial_extended": HOUSE_TIME + SEA + SPATIAL,
}

SUPPLEMENTARY_FEATURE_GROUPS = {
    "A_location_control_with_dong": HOUSE_TIME + DONG,
    "B_spatial_location_control_with_dong": HOUSE_TIME + DONG + SEA + SPATIAL,
}

FEATURE_GROUPS = {**MAIN_FEATURE_GROUPS, **SUPPLEMENTARY_FEATURE_GROUPS}

for group_name, cols in FEATURE_GROUPS.items():
    label = "main" if group_name in MAIN_FEATURE_GROUPS else "supplementary"
    print(f"{group_name}: {len(cols)} raw features ({label})")

## 12. Modeling Functions

In [ ]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def split_feature_types(cols):
    categorical = [c for c in cols if c in CATEGORICAL_COLS]
    binary = [c for c in cols if c in BINARY_COLS]
    numeric = [c for c in cols if c not in set(categorical + binary)]
    return numeric, categorical, binary


def make_preprocessor(cols):
    numeric, categorical, binary = split_feature_types(cols)
    transformers = []
    if numeric:
        transformers.append(("num", StandardScaler(), numeric))
    if categorical:
        transformers.append(("cat", make_one_hot_encoder(), categorical))
    if binary:
        transformers.append(("bin", "passthrough", binary))
    return ColumnTransformer(transformers=transformers, remainder="drop")


def make_models():
    return {
        "Ridge": Ridge(alpha=10.0),
        "RandomForest": RandomForestRegressor(
            n_estimators=300,
            max_depth=20,
            min_samples_leaf=2,
            random_state=RND,
            n_jobs=-1,
        ),
        "XGBoost": XGBRegressor(
            n_estimators=600,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="reg:squarederror",
            tree_method="hist",
            random_state=RND,
            n_jobs=-1,
        ),
    }


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def evaluate_group(data, group_name, cols, log_target=False, add_contract_type=False):
    use_cols = list(cols)
    if add_contract_type and "전월세구분" not in use_cols:
        use_cols.append("전월세구분")

    X = data[use_cols].copy()
    y = data[TARGET].copy()
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RND
    )

    rows = []
    fitted = {}
    for model_name, model in make_models().items():
        pipe = Pipeline([
            ("pre", make_preprocessor(use_cols)),
            ("model", model),
        ])
        estimator = pipe
        if log_target:
            estimator = TransformedTargetRegressor(
                regressor=pipe,
                func=np.log1p,
                inverse_func=np.expm1,
            )

        cv = KFold(n_splits=5, shuffle=True, random_state=RND)
        cv_scores = cross_val_score(
            estimator, X_train, y_train,
            scoring="neg_root_mean_squared_error",
            cv=cv,
            n_jobs=-1,
        )

        estimator.fit(X_train, y_train)
        pred = estimator.predict(X_test)

        rows.append({
            "experiment": group_name,
            "model": model_name,
            "n_rows": len(data),
            "n_features_raw": len(use_cols),
            "log_target": log_target,
            "contract_type_feature": add_contract_type,
            "cv_rmse": float(-cv_scores.mean()),
            "test_rmse": rmse(y_test, pred),
            "test_mae": float(mean_absolute_error(y_test, pred)),
            "test_r2": float(r2_score(y_test, pred)),
        })
        fitted[model_name] = {
            "estimator": estimator,
            "X_test": X_test,
            "y_test": y_test,
            "pred": pred,
            "features": use_cols,
        }
    return pd.DataFrame(rows), fitted

## 13. Main and Supplementary Experiments

The first two rows of `FEATURE_GROUPS` are the original Group A/B comparison. The two `location_control` rows are supplementary diagnostics and should not be described as the original methodology.

In [ ]:
all_results = []
all_fitted = {}

for group_name, cols in FEATURE_GROUPS.items():
    result, fitted = evaluate_group(df, group_name, cols)
    all_results.append(result)
    all_fitted[group_name] = fitted

results = pd.concat(all_results, ignore_index=True)
results = results.sort_values(["experiment", "test_rmse"])
display(results)

results.to_csv(OUT_DIR / "model_results.csv", index=False, encoding="utf-8-sig")

## 14. Performance Comparison

In [ ]:
best_by_group = (
    results.sort_values("test_rmse")
    .groupby("experiment", as_index=False)
    .first()[["experiment", "model", "cv_rmse", "test_rmse", "test_mae", "test_r2"]]
)
display(best_by_group)

rmse_pivot = results.pivot_table(index="model", columns="experiment", values="test_rmse")
display(rmse_pivot)

def compare_delta(left, right):
    merged = results[results["experiment"].isin([left, right])].pivot_table(
        index="model", columns="experiment", values=["test_rmse", "test_mae", "test_r2"]
    )
    delta = pd.DataFrame(index=merged.index)
    delta["delta_rmse"] = merged[("test_rmse", right)] - merged[("test_rmse", left)]
    delta["delta_mae"] = merged[("test_mae", right)] - merged[("test_mae", left)]
    delta["delta_r2"] = merged[("test_r2", right)] - merged[("test_r2", left)]
    return delta

print("Main methodology comparison: Group B - Group A")
display(compare_delta("A_baseline", "B_spatial_extended"))

print("Supplementary location-control comparison: Spatial with dong - Baseline with dong")
display(compare_delta("A_location_control_with_dong", "B_spatial_location_control_with_dong"))

## 15. Prediction Plot for Best Main Spatial Model

The plot compares actual and predicted Jeonse-equivalent deposits for the best model within the main spatial feature set, `B_spatial_extended`.

In [ ]:
best_spatial_model = (
    results[results["experiment"].eq("B_spatial_extended")]
    .sort_values("test_rmse")
    .iloc[0]["model"]
)
r = all_fitted["B_spatial_extended"][best_spatial_model]

plt.figure(figsize=(6, 6))
plt.scatter(r["y_test"], r["pred"], s=10, alpha=0.45, color="#2563eb")
lim = max(r["y_test"].max(), np.max(r["pred"]))
plt.plot([0, lim], [0, lim], "r--", lw=1)
plt.xlabel("Actual Jeonse-equivalent deposit (10,000 KRW)")
plt.ylabel("Predicted Jeonse-equivalent deposit (10,000 KRW)")
plt.title(f"Prediction vs Actual: {best_spatial_model}")
plt.tight_layout()
plt.savefig(OUT_DIR / "prediction_vs_actual_best_main_spatial.png", dpi=150)
plt.show()

## 16. XGBoost Feature Importance

The feature-importance plot is generated for the main spatial model, `B_spatial_extended`. It should be interpreted as model-based association, not causal evidence.

In [ ]:
group_name = "B_spatial_extended"
model_name = "XGBoost"
bundle = all_fitted[group_name][model_name]
estimator = bundle["estimator"]
pipe = estimator.regressor_ if hasattr(estimator, "regressor_") else estimator
pre = pipe.named_steps["pre"]
model = pipe.named_steps["model"]

feature_names = pre.get_feature_names_out()
importance = pd.DataFrame({
    "feature": feature_names,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False)

display(importance.head(20))
importance.to_csv(OUT_DIR / "xgboost_feature_importance.csv", index=False, encoding="utf-8-sig")

top = importance.head(15).iloc[::-1]
plt.figure(figsize=(9, 6))
plt.barh(top["feature"], top["importance"], color="#059669")
plt.title("XGBoost Feature Importance: Main Spatial Model")
plt.tight_layout()
plt.savefig(OUT_DIR / "xgboost_feature_importance_top15.png", dpi=150)
plt.show()

## 17. Supplementary Robustness Checks

These checks are not in the original minimum methodology. They are included to prevent hidden logical dependence on one modeling choice:

1. **Log target:** checks sensitivity to the long-tail distribution of deposits.
2. **Contract type feature:** checks whether converted monthly-rent transactions require an explicit transaction-type control.
3. **Jeonse-only sample:** checks whether conclusions depend on monthly-rent conversion.

In [ ]:
robust_results = []

r, _ = evaluate_group(
    df,
    "B_spatial_extended_log_target",
    FEATURE_GROUPS["B_spatial_extended"],
    log_target=True,
)
robust_results.append(r)

r, _ = evaluate_group(
    df,
    "B_spatial_extended_contract_type",
    FEATURE_GROUPS["B_spatial_extended"],
    add_contract_type=True,
)
robust_results.append(r)

jeonse_only = df[df["전월세구분"].eq("전세")].copy()
r, _ = evaluate_group(
    jeonse_only,
    "B_spatial_extended_jeonse_only",
    FEATURE_GROUPS["B_spatial_extended"],
)
robust_results.append(r)

robust = pd.concat(robust_results, ignore_index=True).sort_values(["experiment", "test_rmse"])
display(robust)
robust.to_csv(OUT_DIR / "robustness_results.csv", index=False, encoding="utf-8-sig")

## 18. Optional SHAP Analysis

SHAP is optional because it can take longer to run. Set `RUN_SHAP = True` after confirming that the main experiment runs successfully.

In [ ]:
RUN_SHAP = False

if RUN_SHAP:
    if importlib.util.find_spec("shap") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "shap"])
    import shap

    X_test_transformed = pre.transform(bundle["X_test"])
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test_transformed)

    shap.summary_plot(
        shap_values,
        X_test_transformed,
        feature_names=feature_names,
        max_display=20,
    )

## 19. Interpretation Guide

Use the results as follows:

1. The original hypothesis is evaluated by comparing `B_spatial_extended` against `A_baseline`.

2. If `B_spatial_extended` has lower RMSE/MAE and higher R² than `A_baseline`, the result supports the claim that spatial-derived variables add predictive power.

3. If the improvement is small or inconsistent across models, the study should report that additional predictive power was limited under this dataset and model setup.

4. The `location_control_with_dong` comparison is supplementary. If adding `읍면동` reduces the marginal effect of spatial variables, this suggests overlap between administrative location information and spatial-derived variables.

5. Feature importance should be interpreted as model-based association, not causal evidence that schools, parks, or sea proximity directly cause higher Jeonse prices.

6. If Jeonse-only results differ from converted-all-transaction results, the monthly-rent conversion assumption must be discussed as a limitation.

Conservative final statement template:

> This study tests whether spatial-derived variables improve Jeonse-equivalent deposit prediction in Buk-gu, Pohang. The main evidence comes from the Group A/B performance comparison. Supplementary checks using administrative dong controls, Jeonse-only samples, and alternative target transformations are used only to assess robustness, not to redefine the original research question.